In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import catboost
import os
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score, precision_score, accuracy_score

In [2]:
DataFileName = "datasets/train.csv" 
df = pd.read_csv(DataFileName)
M, N = df.shape
categorical_features = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

# first, find all potentially invalid data and outliers
borders = {"sleep_duration": [0.0, 24.0], "heart_rate": [0.0, 220.0], "bmi": [0.0, 100],
           "calorie_expenditure": [0.0, np.inf], "step_count": [0.0, np.inf], 
           "exercise_duration": [0.0, 1440.0], "water_intake": [0.0, np.inf],
           "diet_type": ("veg", "non-veg", "balanced"), "stress_level": ("low", "high", "medium"),
           "sleep_quality": ('average', 'poor', 'good'), "physical_activity_level": ('sedentary', 'moderate', 'active'),
           "smoking_alcohol": ('yes', 'occasional', 'no'), "gender": ('female', 'other', 'male')}

for col_name in df.columns:
    if col_name not in borders:
        continue
    else:
        try:
            if type(borders[col_name]) is tuple:
                mask = ~(df[col_name].isin(borders[col_name]) | df[col_name].isna())
            elif type(borders[col_name]) is list:
                mask = ~(((df[col_name] >= borders[col_name][0]) & (df[col_name] <= borders[col_name][1])) | df[col_name].isna())
            else:
                raise AttributeError("Non-standard data type in the borders dict")
        except AttributeError as ae:
            print(f"borders dict is corrupted: {ae}")
        print(f"Column {col_name}: {mask.sum()}")
        
            

Column sleep_duration: 0
Column heart_rate: 0
Column bmi: 0
Column calorie_expenditure: 0
Column step_count: 0
Column exercise_duration: 0
Column water_intake: 0
Column diet_type: 0
Column stress_level: 0
Column sleep_quality: 0
Column physical_activity_level: 0
Column smoking_alcohol: 0
Column gender: 0


### Conclusion
All values lie within valid ranges

In [3]:
missing_per_row = df.isnull().sum(axis=1)
missing_counts = [[num, np.round(num/M, 2)] for num in missing_per_row.value_counts().sort_index()]
print("Number of rows by missing-value count (count, share of the whole df):")
print(*missing_counts, sep="\n")

Number of rows by missing-value count (count, share of the whole df):
[349623, np.float64(0.51)]
[248134, np.float64(0.36)]
[77311, np.float64(0.11)]
[13445, np.float64(0.02)]
[1479, np.float64(0.0)]
[87, np.float64(0.0)]
[9, np.float64(0.0)]


In [4]:
# since the overall share of samples with 3+ missing values is <= 0.2, we drop them
df = df[~(df.isnull().sum(axis=1) >= 3)]

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

df[categorical_features] = df[categorical_features].fillna("Unknown").astype(str)

le = LabelEncoder()
df["health_condition"] = le.fit_transform(df["health_condition"])

# 3. Split into train / test
y = df["health_condition"]
df_train, df_test = train_test_split(df, stratify=y, random_state=42, test_size=0.2)

X_train = df_train.drop(columns=["health_condition", "id"])
y_train = df_train["health_condition"] 

X_test = df_test.drop(columns=["health_condition", "id"])
y_test = df_test["health_condition"] 

model = CatBoostClassifier(
    iterations=1000,
    loss_function='MultiClass',       # FIXED: 'MultiClass' instead of 'Logloss'
    eval_metric='MultiClass',         # Or 'MultiLogloss'
    early_stopping_rounds=50,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    verbose=100,
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.122455
0:	learn: 0.9089614	test: 0.9089835	best: 0.9089835 (0)	total: 1.37s	remaining: 22m 45s
100:	learn: 0.0889039	test: 0.0880617	best: 0.0880617 (100)	total: 1m 55s	remaining: 17m 5s
200:	learn: 0.0865980	test: 0.0865790	best: 0.0865790 (200)	total: 3m 41s	remaining: 14m 39s
300:	learn: 0.0853162	test: 0.0860200	best: 0.0860200 (300)	total: 5m 27s	remaining: 12m 40s
400:	learn: 0.0842012	test: 0.0858226	best: 0.0858226 (400)	total: 7m 18s	remaining: 10m 55s
500:	learn: 0.0831625	test: 0.0856909	best: 0.0856909 (500)	total: 9m 8s	remaining: 9m 6s
600:	learn: 0.0823324	test: 0.0856396	best: 0.0856382 (599)	total: 10m 57s	remaining: 7m 16s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.08562735347
bestIteration = 609

Shrink model to first 610 iterations.


CatBoostClassifier(early_stopping_rounds=50, eval_metric='MultiClass', iterations=1000, loss_function='MultiClass', verbose=100)

In [18]:
if not os.path.exists("submission"):
    os.makedirs("submission")

df = pd.read_csv("datasets/test.csv")
y_id = df["id"]
df.drop(columns=["id"], inplace=True)
df[categorical_features] = df[categorical_features].fillna("Unknown").astype(str)

y_pred = le.inverse_transform(model.predict(df))
df_pred = pd.DataFrame({"health_condition": y_pred}, index=y_id)
df_pred.to_csv("submission/grad_bust_1.csv")


c:\Users\user_1\miniforge3\envs\ipynb_base\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
